# 03 · Compare & Serve

**Phase notebook** — feature `004-notebook-split`. Consumes `lora_adapter/` + `chroma_index/`; serves the three-panel Gradio demo. The ONLY notebook that co-locates the full stack.
Run top-to-bottom on a fresh Colab runtime. All constants come from the shared `config.py`.

## Deviations
- **Notebook decomposition (constitution v1.2.0 §I / §Notebook Decomposition).** One of four phase
  notebooks split from the original single `notebook.ipynb`, for modularity/reuse and dependency-conflict
  isolation. Phases hand off **only** via Drive artifacts; the single shared `config.py` is the only
  cross-notebook import. This supersedes the single-notebook risk-first stage order (recorded per §Governance).


In [ ]:
# \u2500\u2500 Cell 1 \u00b7 Bootstrap: fetch shared config.py, install THIS phase's deps \u2500\u2500
# The four phase notebooks share exactly ONE module (config.py), fetched from the
# repo (constitution v1.2.0 \u00a7Notebook Decomposition). Set REPO_URL to your repo.
# For a PRIVATE repo, use a token URL (https://<TOKEN>@github.com/...) or the
# raw-download fallback shown below.
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_ORG/fine-tuning-demo.git"  # TODO: set to your repo
REPO_DIR = "/content/fine-tuning-demo"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
# Fallback (config.py only, e.g. private repo without clone access):
#   from urllib.request import urlretrieve
#   urlretrieve("<raw-url>/config.py", "/content/config.py"); sys.path.insert(0, "/content")

import config  # the ONLY cross-notebook import (FR-012)

# Serve phase: the FULL stack (the only notebook that co-locates everything).
PINNED_REQS = [
    "unsloth",
    "unsloth_zoo",
    "transformers>=5.2",
    "peft>=0.14",
    "bitsandbytes>=0.45",
    "sentence-transformers>=5.2",
    "chromadb>=1.0",
    "gradio>=5",
    "pillow>=10.4,<12",
]

config.mount_drive()
config.install_deps(PINNED_REQS)
# vLLM is NOT needed for SFT + HF .generate() and hard-caps transformers<5 (Qwen3.5
# needs >=5). Remove it if any dep pulled it in, so the full stack can coexist.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], check=False)
config.set_seeds()

In [ ]:
# ── Cell 2 · Reload pipeline from Drive artifacts (ported from monolith Stage 6) ──
import torch, chromadb
from unsloth import FastLanguageModel
from peft import PeftModel
from sentence_transformers import SentenceTransformer

# Resolve GPU profile; MUST match the base the adapter was trained on (checked below)
config.select_profile(config.detect_vram_gb())

ADAPTER_PATH = config.lora_adapter_path()
CHROMA_PATH  = config.chroma_index_path()

# Fail fast on MISSING or DRIFTED artifacts BEFORE loading anything (FR-007 / FR-013)
config.verify_meta(ADAPTER_PATH, {"base_model_id": config.MODEL_ID})
config.verify_meta(CHROMA_PATH,  {"embed_model":   config.EMBED_MODEL})

# Reload base model, then attach the LoRA adapter.
# FastLanguageModel.for_inference() enables unsloth's fast inference kernels —
# must be called before .generate() on an unsloth-loaded model.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.MODEL_ID, max_seq_length=config.MAX_SEQ_LEN, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)

peft_model = PeftModel.from_pretrained(model, ADAPTER_PATH)
peft_model.eval()
FastLanguageModel.for_inference(peft_model)
print(f"LoRA adapter loaded from: {ADAPTER_PATH}")

# Reconnect RAG (Chroma + embedder)
_chroma     = chromadb.PersistentClient(path=CHROMA_PATH)
collection  = _chroma.get_collection("domain_docs")
embed_model = SentenceTransformer(config.EMBED_MODEL)
print(f"ChromaDB reconnected: {collection.count()} chunks")

def retrieve(query, k=config.TOP_K):
    q_emb   = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    return "\n---\n".join(results["documents"][0] if results["documents"] else [])

In [ ]:
# ── Cell 3 · Shared generation + the four panels (§IV Fair Comparison) ──
# All four panels route through ONE _generate() so base model, quantization, system
# prompt, and generation params are IDENTICAL. The ONLY differences are the adapter
# (peft_model vs model) and the retrieved context (§IV / FR-005). Every value from config.
import traceback

def _content(text):
    # `tokenizer` here is the model's full multimodal Processor (Qwen3.5/Qwen3-VL family
    # are VLM-capable). Its apply_chat_template() pre-scans message["content"] for embedded
    # images/videos and assumes a content-parts list; a plain string gets iterated character
    # by character, raising "string indices must be integers, not 'str'". Wrapping every
    # message in this list-of-parts form works on both old and new transformers/processors.
    return [{"type": "text", "text": text}]

def _generate(mdl, messages):
    # enable_thinking is a Qwen3-family feature; guard against older tokenizer builds.
    # Only swallow the TypeError if it's actually about the unsupported kwarg, otherwise
    # re-raise — else a different bug would silently repeat on retry and get masked.
    try:
        ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            enable_thinking=config.ENABLE_THINKING, return_tensors="pt").to("cuda")
    except TypeError as e:
        if "enable_thinking" not in str(e):
            raise
        ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(
            ids, max_new_tokens=config.MAX_NEW_TOKENS, temperature=config.TEMPERATURE,
            top_p=config.TOP_P, do_sample=True, repetition_penalty=config.REPETITION_PENALTY)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def infer_base(question):
    # Standard panel — base model, no adapter, no RAG
    try:
        messages = [{"role": "system", "content": _content(config.SYSTEM_PROMPT)},
                    {"role": "user",   "content": _content(question)}]
        return _generate(model, messages), ""
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return f"⚠️ Generation error: {type(e).__name__}: {e}", tb

def infer_rag_only(question):
    # Standard + Docs panel — base model + RAG, no adapter
    try:
        prompt = config.RETRIEVAL_TEMPLATE.format(context=retrieve(question), question=question)
        messages = [{"role": "system", "content": _content(config.SYSTEM_PROMPT)},
                    {"role": "user",   "content": _content(prompt)}]
        return _generate(model, messages), ""
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return f"⚠️ Generation error: {type(e).__name__}: {e}", tb

def infer_specialized_no_rag(question):
    # Specialized (No RAG) — LoRA adapter alone, no retrieved context. Isolates what the
    # fine-tune learned from the training QA pairs vs what RAG adds on top of it.
    try:
        messages = [{"role": "system", "content": _content(config.SYSTEM_PROMPT)},
                    {"role": "user",   "content": _content(question)}]
        return _generate(peft_model, messages), ""
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return f"⚠️ Generation error: {type(e).__name__}: {e}", tb

def infer_specialized(question):
    # Specialized (RAG) panel — LoRA adapter + RAG
    try:
        prompt = config.RETRIEVAL_TEMPLATE.format(context=retrieve(question), question=question)
        messages = [{"role": "system", "content": _content(config.SYSTEM_PROMPT)},
                    {"role": "user",   "content": _content(prompt)}]
        return _generate(peft_model, messages), ""
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return f"⚠️ Generation error: {type(e).__name__}: {e}", tb

print("✅ infer_base / infer_rag_only / infer_specialized_no_rag / infer_specialized ready")

In [ ]:
# ── Cell 4 · Gradio four-panel demo (ported from monolith Stage 7; +No-RAG panel, +timings) ──
import gradio as gr
import time
gr.close_all()

def _run_all(question):
    t0 = time.perf_counter()
    std_ans, std_tb = infer_base(question)
    t1 = time.perf_counter()
    rag_ans, rag_tb = infer_rag_only(question)
    t2 = time.perf_counter()
    spec_norag_ans, spec_norag_tb = infer_specialized_no_rag(question)
    t3 = time.perf_counter()
    spec_ans, spec_tb = infer_specialized(question)
    t4 = time.perf_counter()

    # Panels run sequentially on the same GPU (one model call at a time), so each
    # delta below is that panel's own wall-clock time, not a shared/overlapping one.
    timings = {
        "Standard": t1 - t0,
        "Standard + Docs": t2 - t1,
        "Specialized (No RAG)": t3 - t2,
        "Specialized (RAG)": t4 - t3,
    }
    fastest = min(timings, key=timings.get)
    timing_str = "  |  ".join(f"{name}: {secs:.2f}s" for name, secs in timings.items())
    timing_str += f"   →  fastest: {fastest}"

    tracebacks = [t for t in (std_tb, rag_tb, spec_norag_tb, spec_tb) if t]
    debug_log = "\n\n".join(tracebacks) if tracebacks else "(no errors)"
    return std_ans, rag_ans, spec_norag_ans, spec_ans, timing_str, debug_log

with gr.Blocks(title="Domain LLM POC — Full Demo") as demo:
    gr.Markdown("## Domain LLM POC — Full Comparison")
    gr.Markdown(
        "| Panel | Model | Adapter | RAG |\n"
        "|---|---|---|---|\n"
        "| Standard Model | Base | ✗ | ✗ |\n"
        "| Standard + Docs | Base | ✗ | ✓ |\n"
        "| Specialized (No RAG) | Base | ✓ | ✗ |\n"
        "| Specialized (RAG) | Base | ✓ | ✓ |")
    with gr.Row():
        q_in = gr.Textbox(label="Your question", lines=3, placeholder="Type a domain question…", scale=4)
    with gr.Row():
        btn = gr.Button("Submit", variant="primary")
    out_timing = gr.Textbox(label="⏱ Timings", interactive=False, lines=1)
    with gr.Row():
        with gr.Column():
            out_std  = gr.Textbox(label="Standard Model",       interactive=False, lines=14)
        with gr.Column():
            out_rag  = gr.Textbox(label="Standard + Docs",      interactive=False, lines=14)
        with gr.Column():
            out_spec_norag = gr.Textbox(label="Specialized (No RAG)", interactive=False, lines=14)
        with gr.Column():
            out_spec = gr.Textbox(label="Specialized (RAG)",    interactive=False, lines=14)
    with gr.Accordion("Debug log (full traceback on error)", open=False):
        out_debug = gr.Textbox(label="Traceback", interactive=False, lines=20)
    btn.click(fn=_run_all, inputs=q_in,
              outputs=[out_std, out_rag, out_spec_norag, out_spec, out_timing, out_debug])

demo.launch(share=True, server_name="0.0.0.0", debug=True)
print("✅ Full demo UI launched — share URL shown above")